# Phase 02 — Huấn luyện E-GraphSAGE binary / multiclass trên Colab

Chạy sau khi đã có artifact Phase 01 (`processed_v2`) trên Drive (xem `02_phase01_cse_cic_etl_graph.ipynb`). Chọn **Runtime → GPU (T4)** trước khi chạy.

- Dữ liệu đồ thị được copy sang đĩa cục bộ `/content` (đọc shard nhanh); **checkpoint ghi thẳng lên Drive** (`--runs-dir`) nên phiên bị ngắt vẫn **resume** được (model + optimizer + scheduler + best + epoch). Chạy lại cell train là tự tiếp tục.
- Validation tính theo **flow duy nhất**. Chọn checkpoint theo F1-macro trên val. **Không chạy `--eval-test` cho tới khi cấu hình đã khóa.**
- Mỗi cấu hình ghi vào thư mục riêng `graphsage_<loss>_seed<seed>` — không ghi đè nhau.

In [ ]:
import os, sys, shutil
GITHUB_USER, GITHUB_REPO = "thunww", "gnn-nids-tttn"
DRIVE_ROOT = "/content/drive/MyDrive/GNN-NIDS_TTTN"
PROJECT_DIR = "/content/GNN-NIDS_TTTN"
DATASET = "nf-cse-cic-ids2018-v2"        # hoac nf-unsw-nb15-v2
DRIVE_PROCESSED = f"{DRIVE_ROOT}/data/processed_v2"
LOCAL_PROCESSED = "/content/processed_v2"
RUNS_DIR = f"{DRIVE_ROOT}/runs"           # checkpoint + ket qua (ben vung tren Drive)

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
if os.path.isdir(PROJECT_DIR):
    !cd {PROJECT_DIR} && git pull
else:
    !git clone https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git {PROJECT_DIR}
!pip install -q -r {PROJECT_DIR}/requirements.txt
!nvidia-smi -L
import torch; print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

In [ ]:
# Copy artifact (graphs + contract + novel ids) sang dia cuc bo -- bo qua flows.parquet/raw_test cho nhe
src, dst = f"{DRIVE_PROCESSED}/{DATASET}", f"{LOCAL_PROCESSED}/{DATASET}"
if os.path.isdir(dst):
    print("Da co ban sao cuc bo")
else:
    shutil.copytree(src, dst, ignore=shutil.ignore_patterns("flows.parquet", "raw_test.parquet", "models"))
print(os.listdir(dst))

## 1. Kiểm tra nhanh (test + 2 epoch trên vài shard)

In [ ]:
!cd {PROJECT_DIR} && PYTHONPATH=src python -m pytest src -q -p no:cacheprovider 2>&1 | tail -2
!cd {PROJECT_DIR} && PYTHONPATH=src python src/models/train_gnn_v2.py --dataset {DATASET} --task multiclass --processed {LOCAL_PROCESSED} --runs-dir /content/smoke_runs --epochs 2 --max-train-shards 2 --fresh

## 2. Pilot đo tài nguyên

Chạy 3–5 epoch để đo thời gian/epoch và VRAM, rồi quyết định `--epochs` và `--batch-size` (thử 8/16/32). Nếu OOM thì giảm batch, **không bỏ lớp hay bỏ test**.

In [ ]:
!cd {PROJECT_DIR} && PYTHONPATH=src python src/models/train_gnn_v2.py --dataset {DATASET} --task multiclass --processed {LOCAL_PROCESSED} --runs-dir /content/pilot_runs --epochs 3 --fresh
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

## 3. Train đầy đủ

Chạy lại cell này sau khi bị ngắt: tự resume từ `last.pt` trên Drive. Mỗi cấu hình/seed là một cell độc lập. Khi dùng nhiều seed, báo cáo mean ± std trên cùng split; nếu chỉ chạy một lần, ghi rõ là single-run.

In [ ]:
TASK, LOSS, SEED = "multiclass", "cb", 42      # task: binary | multiclass ; loss: ce | cb | weighted
!cd {PROJECT_DIR} && PYTHONPATH=src python src/models/train_gnn_v2.py --dataset {DATASET} --task {TASK} --processed {LOCAL_PROCESSED} --runs-dir {RUNS_DIR} --loss {LOSS} --seed {SEED}

In [ ]:
import pandas as pd
run = f"{RUNS_DIR}/{DATASET}/{TASK}/graphsage_{LOSS}_seed{SEED}"
display(pd.read_csv(f"{run}/history.csv").tail(5))
display(pd.read_csv(f"{run}/val_confusion_normalized.csv", index_col=0).round(2))

## 4. Đánh giá TEST — chỉ chạy một lần, khi cấu hình đã khóa

Đầu ra: `test_metrics.json` (toàn test và riêng "flow mới"), `test_predictions.parquet` theo `flow_id`, ma trận nhầm lẫn raw/normalized. Không chọn seed hay cấu hình dựa trên kết quả test.

In [ ]:
# !cd {PROJECT_DIR} && PYTHONPATH=src python src/models/train_gnn_v2.py --dataset {DATASET} --task {TASK} --processed {LOCAL_PROCESSED} --runs-dir {RUNS_DIR} --loss {LOSS} --seed {SEED} --eval-test